# PVG baselines (v5 setup + Brandon's baseline cell, 28 Oct)

Scores the rule check and three untrained Qwen judges (1.5B, 7B, 32B in 4-bit) on the same held-out rows the PVG verifier was scored on, with the verifier's own prompt and SOUND/UNSOUND log-prob scoring. No training.

Runtime: A100 (the 32B judge needs about 20 GB in 4-bit). About 15 minutes of setup (the setup cell regenerates the Qwen-written held-out rows), then a few minutes per judge. Results go to `eval/baselines_v5_all.json` and `MyDrive/pvg_v5_results/baselines_v5_all.json`.

The Qwen-written rows are sampled again here, so `heldout_100` may differ by a few rows from the 52 used in the v5 run.

## 1. Setup

In [ ]:
import os
from pathlib import Path

REPO_URL  = "https://github.com/gunasti2002/Automated_R_leg_NoAPI.git"
BRANCH    = "main"
CLONE_DIR = Path("/content/Automated_R_leg_NoAPI")

%cd /content
!rm -rf {CLONE_DIR}
!git clone --branch {BRANCH} {REPO_URL} {CLONE_DIR}

matches = sorted(CLONE_DIR.rglob("config.py"))
assert matches, f"No config.py found under {CLONE_DIR}"
PROJECT_ROOT = matches[0].parent
os.chdir(PROJECT_ROOT)
print("Project root:", PROJECT_ROOT)
!git -C {CLONE_DIR} log --oneline -3


In [ ]:
# Colab ships torchao 0.10, which peft refuses (needs >= 0.16). Nothing here uses it: remove it.
!pip uninstall -y -q torchao
# torch + matplotlib ship with Colab; do not reinstall torch (CUDA wheel mismatch risk).
!pip install -q jinja2 "transformers>=4.50" peft accelerate
import importlib; importlib.invalidate_caches()

import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
!nvidia-smi --query-gpu=name,memory.total,memory.used --format=csv

# bitsandbytes for the 4-bit 32B judge
!pip -q install -U bitsandbytes

### Brandon's fixes

Pins the project code to 51940d1, patches it, rebuilds the dataset with the 10 real records and writes the Qwen-written rows (GPU, about 15 minutes). In the output, check that every patch line starts with `+` and that both model-written files have rows. The data cell below rebuilds the same dataset, which is harmless.

In [ ]:
!python apply_fixes.py
!git -C {CLONE_DIR} status --short | head -20

In [ ]:
# Brandon's change 3: a stronger learning signal for the prover. Subprocesses inherit this.
import os, json
OVERRIDES = {
    "prover_lr": 1e-4,                 # was 2e-5
    "prover_num_generations": 8,       # was 4 samples per record
    "findings_per_round": 16,          # was 8 records per round
    "prover_kl_penalty": 0.05,         # was 0.02
    # Both roles gated by the rule check: helpful earns P(sound) only if it passes,
    # sneaky only if it fails. Matches the verifier labels (sneaky write-ups that pass
    # the rule check are dropped from verifier training, so they should earn nothing).
    "prover_reward_mode": "correctness_gated",
    # v5: 3 warm-start epochs (apply_fixes default is 2; v4 reached helpful fidelity 0.27-0.41)
    "prover_sft_epochs": 3,
}
os.environ["PVG_CONFIG_OVERRIDES"] = json.dumps(OVERRIDES)

import sys; sys.path.insert(0, ".")
from config import PVGConfig
cfg = PVGConfig()
print({k: getattr(cfg, k) for k in OVERRIDES})
print("samples per round:", 2 * cfg.findings_per_round * cfg.prover_num_generations)


## 2. Data

120 honest/sneaky pairs, 20% held out by record, seed 42 (unchanged).

In [ ]:
!python data/build_findings_dataset.py --target-pairs 120 --seed 42 \
    --out data/findings_pvg_dataset.jsonl --heldout-out data/findings_pvg_heldout.jsonl \
    --manifest-out data/dataset_manifest.json | tail -4


## Baselines

In [ ]:
# Baselines on the same held-out rows the PVG verifier was scored on (Brandon, 28 Oct).
# Methods: rule check, zero-shot Qwen2.5-1.5B (the verifier's base model, untrained), and zero-shot
# Qwen2.5-7B and Qwen2.5-32B judges. All model judges use the verifier's own prompt and the same
# SOUND/UNSOUND log-prob scoring, with no LoRA and no calibration bias.
# Eval sets:
#   heldout_100 = 48 original held-out rows + 52 model-written held-out rows (what "heldout_acc" in the training logs used)
#   heldout_48  = the original 48 rows only
#   spot_20     = the 20 hand-checked spot-check rows
import gc, json, os, shutil, time
from pathlib import Path

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

from config import PVGConfig
from training import pvg_loop
from eval.baselines import rule_check_baseline

# ---- settings ---------------------------------------------------------------
JUDGES = {
    "qwen_1.5b_zero_shot": ("Qwen/Qwen2.5-1.5B-Instruct", False),
    "qwen_7b_judge":       ("Qwen/Qwen2.5-7B-Instruct",   False),  # set True on a 16 GB GPU (T4)
    "qwen_32b_judge":      ("Qwen/Qwen2.5-32B-Instruct",  True),   # 4-bit; needs an A100 or L4
}
SKIP = set()            # e.g. {"qwen_32b_judge"} if you only have a T4
OUT_JSON = Path("eval/baselines_v5_all.json")
DRIVE_COPY = "/content/drive/MyDrive/pvg_v5_results/baselines_v5_all.json"

# ---- data -------------------------------------------------------------------
cfg = PVGConfig()
load = lambda p: [json.loads(l) for l in Path(p).read_text().splitlines() if l.strip()]
mw_path = getattr(cfg, "model_written_heldout_path", "") or "data/findings_pvg_heldout_model_written.jsonl"
assert Path(mw_path).exists(), f"{mw_path} missing: run the setup cells (clone, installs, apply_fixes.py, build dataset) first"
h48 = load(cfg.heldout_dataset_path)
mw = load(mw_path)
SETS = {"heldout_100": h48 + mw, "heldout_48": h48, "spot_20": load(cfg.spot_check_set_path)}
print({k: len(v) for k, v in SETS.items()})


def summarize(ev):
    labels, preds = ev["labels"], [s >= pvg_loop.ACCEPT_THRESHOLD for s in ev["scores"]]
    n_s, n_u = sum(labels), len(labels) - sum(labels)
    return {
        "n": ev["n"], "accuracy": ev["accuracy"], "auroc": ev["auroc"], "accept_rate": ev["accept_rate"],
        "sneaky_reject_rate": sum(1 for p, l in zip(preds, labels) if not l and not p) / max(1, n_u),
        "honest_false_reject_rate": sum(1 for p, l in zip(preds, labels) if l and not p) / max(1, n_s),
        "unparseable_share": ev["unparseable_share"],
        "per_perturbation_reject_rate": dict(sorted(ev["per_perturbation_reject_rate"].items())),
    }


def load_judge(name, four_bit):
    tok = AutoTokenizer.from_pretrained(name)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    kw = {"device_map": "auto"}
    if four_bit:
        kw["quantization_config"] = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                                                       bnb_4bit_compute_dtype=torch.bfloat16)
    else:
        kw["dtype"] = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
    model = AutoModelForCausalLM.from_pretrained(name, **kw).eval()
    for p in model.parameters():
        p.requires_grad_(False)
    return model, tok


results = {s: {} for s in SETS}
for s, rows in SETS.items():
    results[s]["rule_check"] = rule_check_baseline(rows)

for key, (name, four_bit) in JUDGES.items():
    if key in SKIP:
        continue
    t0 = time.time()
    print(f"\nLoading {name} ({'4-bit' if four_bit else '16-bit'}) ...")
    model, tok = load_judge(name, four_bit)
    # optimizer is never used for scoring; a placeholder avoids AdamW on frozen weights
    pvg_loop.register_verifier(key, model, tok, "cuda", cfg, optimizer="unused", logit_bias=0.0)
    for s, rows in SETS.items():
        pvg_loop.reset_scoring_stats()
        t1 = time.time()
        results[s][key] = summarize(pvg_loop.evaluate_verifier_on_rows(rows, cfg, verifier_model=key))
        results[s][key]["seconds_per_finding"] = (time.time() - t1) / len(rows)
    print(f"  done in {time.time() - t0:.0f}s")
    pvg_loop._VERIFIER_CACHE.pop(key, None)
    del model, tok
    gc.collect(); torch.cuda.empty_cache()

TRAINED = {  # PVG verifier, round 7, mean across 5 seeds, from the v5 training logs
    "heldout_100": {"accuracy": 0.952, "auroc": 0.992},
    "spot_20": {"accuracy": 0.900, "auroc": 0.968},
}
for s in SETS:
    print(f"\n### {s} (n={len(SETS[s])})")
    print("| method | accuracy | AUROC | sneaky rejected | honest wrongly rejected | sec/finding |")
    print("|---|---|---|---|---|---|")
    for m, r in results[s].items():
        sec = f"{r['seconds_per_finding']:.3f}" if "seconds_per_finding" in r else "-"
        print(f"| {m} | {r['accuracy']:.3f} | {r['auroc']:.3f} | {r['sneaky_reject_rate']:.3f} | "
              f"{r['honest_false_reject_rate']:.3f} | {sec} |")
    if s in TRAINED:
        t = TRAINED[s]
        print(f"| PVG verifier (round 7) | {t['accuracy']:.3f} | {t['auroc']:.3f} | - | - | - |")

print("\nPer-perturbation reject rate on heldout_100:")
for m, r in results["heldout_100"].items():
    print(f"  {m}: {r['per_perturbation_reject_rate']}")

OUT_JSON.parent.mkdir(parents=True, exist_ok=True)
OUT_JSON.write_text(json.dumps({"sets": {k: len(v) for k, v in SETS.items()}, "results": results}, indent=2))
print(f"\nWritten to {OUT_JSON}")
try:
    from google.colab import drive
    drive.mount("/content/drive")
    Path(DRIVE_COPY).parent.mkdir(parents=True, exist_ok=True)
    shutil.copy(OUT_JSON, DRIVE_COPY)
    print("Copied to", DRIVE_COPY)
except Exception as e:
    print("Drive copy skipped:", e)
